In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer


In [2]:
# Load data
df = pd.read_csv("./data/raw_data.csv")

# Drop rows with null values
df = df.dropna()

# Data Preprocessing

In [3]:
cols_to_drop = [
    'EmployeeCount', 'EmployeeNumber',
    'Over18', 'StandardHours', 'HowToEmploy', 'Incentive', 'Years', 'Year'
]

df = df.drop(columns=cols_to_drop, errors='ignore')


In [4]:
target = 'MonthlyIncome'
y = df[target]
X = df.drop(columns=[target])

In [5]:
categorical_cols_to_encode = [
    'Attrition', 'BusinessTravel', 'Department', 'EducationField',
    'Gender', 'JobRole', 'MaritalStatus'
]

In [6]:
# Label encode categoricals
X_encoded = X.copy()
for col in categorical_cols_to_encode:
    if col in X_encoded.columns:
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

# StandardScaler on ALL columns (everything is numeric after encoding)
scaler = StandardScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(X_encoded), columns=X_encoded.columns)

In [7]:
# Add target back and scale it too
processed_data = X_normalized.copy()
processed_data[target] = StandardScaler().fit_transform(y.values.reshape(-1, 1)).flatten()
processed_data = processed_data.replace([np.inf, -np.inf], np.nan).dropna()
processed_data.to_csv('./data/processed_data.csv', index=False)

print(f"✓ Processed data saved — shape: {processed_data.shape}")
print(f"  All columns scaled. Range check:")
print(processed_data.describe().loc[['mean', 'std']].round(4))

✓ Processed data saved — shape: (1470, 37)
  All columns scaled. Range check:
         Age  Attrition  BusinessTravel  Department  DistanceFromHome  \
mean  0.0000    -0.0000         -0.0000     -0.0000            0.0000   
std   1.0003     1.0003          1.0003      1.0003            1.0003   

      Education  EducationField  EnvironmentSatisfaction  Gender  \
mean     0.0000          0.0000                   0.0000 -0.0000   
std      1.0003          1.0003                   1.0003  1.0003   

      PerformanceIndex  ...  YearsWithCurrManager  RemoteWork  StressRating  \
mean            0.0000  ...                0.0000     -0.0000       -0.0000   
std             1.0003  ...                1.0003      1.0003        1.0003   

      WelfareBenefits  InHouseFacility  ExternalFacility  ExtendedLeave  \
mean           0.0000           0.0000            0.0000        -0.0000   
std            1.0003           1.0003            1.0003         1.0003   

      FlexibleWork  StressSelfRep